# Assignment 5 — Catching Data Leakage Before It Catches You

**Course:** Feature Engineering & MLOps  
**Assignment No.:** 5  
**Topic:** Data Leakage — Target Leakage and Preprocessing Leakage

This assignment demonstrates both kinds of leakage using the supplied customer churn dataset, then fixes the workflow so the final model is honest and deployable.

## Notebook Roadmap

### 4.1 — Target Leakage
1. Load and inspect the customer churn dataset.
2. Confirm the cancellation column is available only for churned customers.
3. Compare the suspicious feature correlation with a legitimate feature.
4. Train Model A with legitimate features only.
5. Train Model B with the leaked features included.
6. Compare the accuracy and ROC-AUC gap.

### 4.2 — Preprocessing Leakage
1. Fit a `StandardScaler` on the full dataset.
2. Split the data and fit a new scaler on training data only.
3. Compare their learned statistics.
4. Rebuild the correct preprocessing using one `Pipeline`.

### 4.3 — The Fix
Train the final model using only legitimate features and a train-only preprocessing pipeline, then confirm that it matches Model A.

In [15]:
# Imports and reproducible configuration

import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
RANDOM_STATE = 42
TEST_SIZE = 0.25
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

## 1. Load the Dataset

The assignment uses `customer_churn_a5.csv`, containing 600 telecom customers.

The dataset has four legitimate predictors, two deliberately leaked columns, and the `churn` target. `customer_id` is an identifier, so it is not used as a modelling feature.

In [16]:
# Locate and load the supplied dataset
candidate_paths = [
    "/content/customer_churn_a5.csv"
]
DATA_PATH = next((path for path in candidate_paths if os.path.exists(path)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "customer_churn_a5.csv was not found. "
        "Place it at data/raw/customer_churn_a5.csv or upload it to Colab."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset path :", DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

Dataset path : /content/customer_churn_a5.csv
Dataset shape: (600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,churn,days_since_cancellation,final_bill_amount
0,20406,13,99.170000,Month-to-month,2,0,NaN,NaN
1,20531,24,66.770000,Month-to-month,1,0,NaN,NaN
2,20307,15,44.030000,One year,1,0,NaN,NaN
3,20391,3,61.080000,Two year,3,0,NaN,NaN
4,20147,8,106.720000,Two year,1,0,NaN,NaN


In [17]:
# Basic dataset inspection

print("Column names:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
missing_summary = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
)
missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
)
display(missing_summary)

print("\nTarget distribution:")
display(df["churn"].value_counts().sort_index().to_frame("count"))

Column names:
['customer_id', 'tenure_months', 'monthly_charges', 'contract_type', 'support_calls', 'churn', 'days_since_cancellation', 'final_bill_amount']

Data types:


,dtype
customer_id,int64
tenure_months,int64
monthly_charges,float64
contract_type,object
support_calls,int64
churn,int64
days_since_cancellation,float64
final_bill_amount,float64



Missing values:


,missing_count,missing_percentage
customer_id,0,0.000000
tenure_months,0,0.000000
monthly_charges,0,0.000000
contract_type,0,0.000000
support_calls,0,0.000000
churn,0,0.000000
days_since_cancellation,451,75.166667
final_bill_amount,451,75.166667



Target distribution:


,count
churn,
0,451
1,149


# 4.1 — Target Leakage: Prove It, Don't Just Assert It

A useful first question for any feature is: **could this value exist at prediction time?**

The assignment identifies `days_since_cancellation` and `final_bill_amount` as leaked because they are only available after a customer has cancelled. The checks below demonstrate that directly from the data.

### 1. Confirm the leaked feature is present exactly for churned customers

The assignment asks for a single programmatic check rather than relying on visual inspection.

In [18]:
# Check that non-null cancellation dates correspond exactly to churn = 1
non_null_cancellation_days = df["days_since_cancellation"].notna().sum()
churned_rows = (df["churn"] == 1).sum()
print("Non-null days_since_cancellation:", non_null_cancellation_days)
print("Rows with churn=1              :", churned_rows)
assert non_null_cancellation_days == churned_rows

print("PASS: the two counts are exactly equal.")

Non-null days_since_cancellation: 149
Rows with churn=1              : 149
PASS: the two counts are exactly equal.


### 2. Compare the suspicious correlation with a legitimate feature

For this check, missing `final_bill_amount` values are filled with `-1`, as required by the assignment.

If a feature is almost directly associated with the target while also representing information that only exists after the outcome, that is a strong warning sign of target leakage.

In [19]:
# Correlation check for the suspicious feature
final_bill_filled = df["final_bill_amount"].fillna(-1)
correlation_table = pd.DataFrame({
    "feature": [
        "final_bill_amount (missing -> -1)",
        "tenure_months",
    ],
    "correlation_with_churn": [
        final_bill_filled.corr(df["churn"]),
        df["tenure_months"].corr(df["churn"]),
    ],
})
display(correlation_table.round(6))

,feature,correlation_with_churn
0,final_bill_amount (missing -> -1),0.924978
1,tenure_months,-0.220718


### 3. Model A — Legitimate features only

The legitimate features are:

- `tenure_months`
- `monthly_charges`
- `contract_type`
- `support_calls`

`contract_type` is categorical, so it is one-hot encoded. The numeric preprocessing and categorical preprocessing are kept inside the same pipeline so that everything is fitted using the training data only.

In [20]:
# Features required for the two target-leakage models

legitimate_features = [
    "tenure_months",
    "monthly_charges",
    "contract_type",
    "support_calls",
]
leaked_features = [
    "days_since_cancellation",
    "final_bill_amount",
]
numeric_features = [
    "tenure_months",
    "monthly_charges",
    "support_calls",
]
categorical_features = [
    "contract_type",
]
target = "churn"
X = df[legitimate_features].copy()
y = df[target].copy()
# One reproducible split is reused for the required comparisons.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"Training rows: {len(X_train)} ({len(X_train) / len(X):.0%})")
print(f"Testing rows : {len(X_test)} ({len(X_test) / len(X):.0%})")

Training rows: 450 (75%)
Testing rows : 150 (25%)


In [21]:
# Version-compatible OneHotEncoder
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

legitimate_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_features,
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", encoder),
        ]),
        categorical_features,
    ),
])

model_a = Pipeline([
    ("preprocessor", legitimate_preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])

model_a.fit(X_train, y_train)
pred_a = model_a.predict(X_test)
prob_a = model_a.predict_proba(X_test)[:, 1]
accuracy_a = accuracy_score(y_test, pred_a)
roc_auc_a = roc_auc_score(y_test, prob_a)
model_a_results = pd.DataFrame({
    "Model": ["Model A — Legitimate features"],
    "Test Accuracy": [accuracy_a],
    "ROC-AUC": [roc_auc_a],
})

display(model_a_results.round(6))

,Model,Test Accuracy,ROC-AUC
0,Model A — Legitimate features,0.753333,0.778283


### 4. Model B — Legitimate features plus leaked features

Now the two post-cancellation features are deliberately added.

The missing values are filled with `-1` before modelling, exactly as specified in the assignment. This model is intentionally demonstrating what happens when information that should not exist at prediction time is allowed into the feature set.

In [22]:
# Add the leaked features and fill their missing values with -1
X_with_leakage = df[legitimate_features + leaked_features].copy()
X_with_leakage[leaked_features] = X_with_leakage[leaked_features].fillna(-1)
# Use the same train/test rows as Model A for a fair comparison.
X_train_leaked = X_with_leakage.loc[X_train.index].copy()
X_test_leaked = X_with_leakage.loc[X_test.index].copy()
leaked_numeric_features = numeric_features + leaked_features
leaked_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        leaked_numeric_features,
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", encoder),
        ]),
        categorical_features,
    ),
])

model_b = Pipeline([
    ("preprocessor", leaked_preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])

model_b.fit(X_train_leaked, y_train)
pred_b = model_b.predict(X_test_leaked)
prob_b = model_b.predict_proba(X_test_leaked)[:, 1]
accuracy_b = accuracy_score(y_test, pred_b)
roc_auc_b = roc_auc_score(y_test, prob_b)
model_comparison = pd.DataFrame({
    "Model": [
        "Model A — Legitimate features",
        "Model B — With leakage",
    ],
    "Test Accuracy": [
        accuracy_a,
        accuracy_b,
    ],
    "ROC-AUC": [
        roc_auc_a,
        roc_auc_b,
    ],
})

display(model_comparison.round(6))

,Model,Test Accuracy,ROC-AUC
0,Model A — Legitimate features,0.753333,0.778283
1,Model B — With leakage,1.000000,1.000000


### 5. Accuracy and ROC-AUC Gap

The same test observations are used for both models, so the difference below comes from adding the leaked information rather than from using a different split.

In [23]:
# Measure the performance gap caused by the leaked features
accuracy_gap = accuracy_b - accuracy_a
roc_auc_gap = roc_auc_b - roc_auc_a
gap_table = pd.DataFrame({
    "Metric": ["Test Accuracy", "ROC-AUC"],
    "Model A": [accuracy_a, roc_auc_a],
    "Model B": [accuracy_b, roc_auc_b],
    "Gap (B - A)": [accuracy_gap, roc_auc_gap],
})

display(gap_table.round(6))

,Metric,Model A,Model B,Gap (B - A)
0,Test Accuracy,0.753333,1.000000,0.246667
1,ROC-AUC,0.778283,1.000000,0.221717


**Interpretation:** Model B looks much stronger because it has access to information created after the churn event. If this model were deployed for brand-new customers who have not cancelled yet, `days_since_cancellation` and `final_bill_amount` would not be available. The model would therefore lose the information responsible for its artificial performance advantage, making the reported test score misleading for real deployment.

# 4.2 — Preprocessing Leakage: The Wrong Order vs. the Right Order

Preprocessing can leak information even when all of the features themselves are legitimate.

The comparison below uses only `tenure_months`, `monthly_charges`, and `support_calls`. First, a scaler is fitted using the complete dataset. Then a second scaler is fitted using only the training data. The learned statistics should be different because the first scaler has seen the test observations.

### 1. Wrong order — fit the scaler on the entire dataset

The scaler below sees all 600 observations before the train/test split is used for the comparison.

In [24]:
# Fit StandardScaler on the ENTIRE dataset
preprocessing_numeric = [
    "tenure_months",
    "monthly_charges",
    "support_calls",
]
X_numeric = df[preprocessing_numeric].copy()
full_scaler = StandardScaler()
full_scaler.fit(X_numeric)
wrong_mean = full_scaler.mean_.copy()
wrong_scale = full_scaler.scale_.copy()
wrong_scaler_table = pd.DataFrame({
    "feature": preprocessing_numeric,
    "mean_": wrong_mean,
    "scale_": wrong_scale,
})

display(wrong_scaler_table.round(6))

,feature,mean_,scale_
0,tenure_months,19.561667,17.524541
1,monthly_charges,65.922017,22.799319
2,support_calls,2.165000,1.498146


### 2. Correct order — split first, then fit on training data only

The same reproducible train/test split is used. The important difference is that the scaler learns its statistics from `X_train` only.

In [25]:
# Fit a NEW StandardScaler on the TRAINING split only
X_train_numeric = X_numeric.loc[X_train.index].copy()
X_test_numeric = X_numeric.loc[X_test.index].copy()
train_scaler = StandardScaler()
train_scaler.fit(X_train_numeric)
correct_mean = train_scaler.mean_.copy()
correct_scale = train_scaler.scale_.copy()
correct_scaler_table = pd.DataFrame({
    "feature": preprocessing_numeric,
    "mean_": correct_mean,
    "scale_": correct_scale,
})

display(correct_scaler_table.round(6))

,feature,mean_,scale_
0,tenure_months,19.895556,17.445158
1,monthly_charges,65.794733,22.263904
2,support_calls,2.117778,1.483807


### 3. Difference between the two scalers

Even if the numerical differences are not large, the full-data scaler has still learned from observations that are supposed to represent unseen test data.

In [26]:
# Compare the means learned from full data vs. training data only
scaler_comparison = pd.DataFrame({
    "feature": preprocessing_numeric,
    "full_data_mean": wrong_mean,
    "train_only_mean": correct_mean,
    "absolute_mean_difference": np.abs(wrong_mean - correct_mean),
})

display(scaler_comparison.round(6))

,feature,full_data_mean,train_only_mean,absolute_mean_difference
0,tenure_months,19.561667,19.895556,0.333889
1,monthly_charges,65.922017,65.794733,0.127283
2,support_calls,2.165000,2.117778,0.047222


The principle matters regardless of the size of the gap. The test set is meant to behave like unseen future data, so preprocessing statistics must not be influenced by it. A small difference on this dataset does not make the workflow safe; on another dataset, a different distribution or unusual test observations could make the effect much larger.

### 4. Correct preprocessing using one `Pipeline`

The pipeline below contains the required **imputer + scaler + model**. It is fitted only on the training split, so the preprocessing statistics are learned in the correct order.

In [27]:
# Build the correct train-only preprocessing + model pipeline
correct_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
        ),
    ),
])
correct_pipeline.fit(X_train_numeric, y_train)
pipeline_scaler = correct_pipeline.named_steps["scaler"]
pipeline_mean = pipeline_scaler.mean_
pipeline_scale = pipeline_scaler.scale_
pipeline_comparison = pd.DataFrame({
    "feature": preprocessing_numeric,
    "manual_train_mean": correct_mean,
    "pipeline_train_mean": pipeline_mean,
    "manual_train_scale": correct_scale,
    "pipeline_train_scale": pipeline_scale,
})
display(pipeline_comparison.round(6))
assert np.allclose(correct_mean, pipeline_mean)
assert np.allclose(correct_scale, pipeline_scale)

print("PASS: Pipeline scaler statistics exactly match the manual train-only scaler.")

,feature,manual_train_mean,pipeline_train_mean,manual_train_scale,pipeline_train_scale
0,tenure_months,19.895556,19.895556,17.445158,17.445158
1,monthly_charges,65.794733,65.794733,22.263904,22.263904
2,support_calls,2.117778,2.117778,1.483807,1.483807


PASS: Pipeline scaler statistics exactly match the manual train-only scaler.


# 4.3 — The Fix

The final model uses only legitimate features and keeps preprocessing inside the model pipeline. The split happens first, and the pipeline is fitted only on the training data.

This is the honest, deployable version of the churn model.

In [28]:
# Final deployable model — legitimate features only
final_model = Pipeline([
    ("preprocessor", legitimate_preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])
final_model.fit(X_train, y_train)
final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]
final_accuracy = accuracy_score(y_test, final_pred)
final_roc_auc = roc_auc_score(y_test, final_prob)
final_results = pd.DataFrame({
    "Model": [
        "Model A — Honest baseline",
        "Final fixed model",
    ],
    "Test Accuracy": [
        accuracy_a,
        final_accuracy,
    ],
    "ROC-AUC": [
        roc_auc_a,
        final_roc_auc,
    ],
})

display(final_results.round(6))
assert np.isclose(final_accuracy, accuracy_a)
assert np.isclose(final_roc_auc, roc_auc_a)
print("PASS: the final fixed model matches Model A.")

,Model,Test Accuracy,ROC-AUC
0,Model A — Honest baseline,0.753333,0.778283
1,Final fixed model,0.753333,0.778283


PASS: the final fixed model matches Model A.


## Final Conclusion

The results show why data leakage can make a model look better than it really is. `days_since_cancellation` and `final_bill_amount` contain post-outcome information, so including them gives Model B an unfair advantage that would not exist when predicting a new customer. Preprocessing leakage is another issue: even legitimate features must not be used to calculate scaling statistics from the test set. Fitting the preprocessing only on the training split prevents that information from crossing the train/test boundary. The final pipeline uses only legitimate features and reproduces the honest Model A results, so it is the version that can be used for real prediction.